# QLoRA Fine-Tuning in Google Colab

This notebook fine-tunes Mistral-7B-Instruct with your capstone dataset using QLoRA.

It will:
1. Verify GPU availability
2. Clone or refresh the repository at branch feature/prompt-generator
3. Install required packages
4. Resolve dataset and script paths from the repo structure
5. Run training
6. Zip and download the adapter output

In [1]:
import subprocess
import torch

print('[+] GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('[+] GPU name:', torch.cuda.get_device_name(0))
    print('[+] CUDA version (torch):', torch.version.cuda)
    print('[+] Device count:', torch.cuda.device_count())
else:
    print('[!] No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU')

print('\n[+] nvidia-smi:')
try:
    result = subprocess.run(['nvidia-smi'], check=False, text=True, capture_output=True)
    print(result.stdout if result.stdout else result.stderr)
except Exception as e:
    print('[!] Could not run nvidia-smi:', e)

[+] GPU available: True
[+] GPU name: Tesla T4
[+] CUDA version (torch): 12.8
[+] Device count: 1

[+] nvidia-smi:
Sat Mar 28 10:14:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                        

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/MeenakshiPathiyil/capstone.git"
BRANCH = "feature/prompt-generator"
TARGET = "/content/capstone"

def run(cmd, cwd=None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if os.path.isdir(os.path.join(TARGET, ".git")):
    print("[i] Existing repository detected, refreshing branch...")
    run("git fetch origin", cwd=TARGET)
    run(f"git checkout {BRANCH}", cwd=TARGET)
    run(f"git pull --ff-only origin {BRANCH}", cwd=TARGET)
else:
    if os.path.isdir(TARGET):
        run(f"rm -rf {TARGET}")
    run(f"git clone --single-branch --branch {BRANCH} {REPO_URL} {TARGET}")

os.chdir("/content/capstone/scenario_generation")
print("[+] Working directory:", os.getcwd())
run("git branch --show-current")
run("wc -l dataset/training_data.jsonl")

In [2]:
from pathlib import Path

req_path = Path('requirements_finetune.txt')
if not req_path.exists():
    req_path.write_text(
        '\n'.join([
            'torch>=2.1.0',
            'transformers>=4.40.0',
            'peft>=0.10.0',
            'trl>=0.8.0',
            'bitsandbytes>=0.43.0',
            'datasets>=2.18.0',
            'accelerate>=0.28.0',
        ]) + '\n',
        encoding='utf-8'
    )

!pip install -q -r requirements_finetune.txt
print('[✓] Installed packages from requirements_finetune.txt')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.8 MB/s eta 0:00:00
[✓] Installed packages from requirements_finetune.txt


In [ ]:
from pathlib import Path
import os
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stderr}")
    return result.stdout.strip()

repo_root = Path("/content/capstone/scenario_generation")
if not repo_root.exists():
    raise FileNotFoundError(
        "Repository not found at /content/capstone/scenario_generation. Run the clone cell first."
    )

os.chdir(repo_root)
print("[+] Current working directory:", Path.cwd())
print("[+] Active git branch:", run("git branch --show-current"))

required = [
    repo_root / "dataset" / "training_data.jsonl",
    repo_root / "fine_tuning" / "fine_tune.py",
]

for p in required:
    if p.exists():
        print(f"[✓] Found: {p}")
    else:
        print(f"[!] Missing: {p}")

print("\n[+] training_data.jsonl line count:")
print(run("wc -l dataset/training_data.jsonl"))

Upload training_data.jsonl first, then fine_tune.py


Saving training_data.jsonl to training_data (1).jsonl
[✓] Uploaded: training_data (1).jsonl

[+] /content files:
['.config', 'finetuning.py', 'training_data (1).jsonl', 'requirements_finetune.txt', 'training_data.jsonl', 'fine_tune.py', 'sample_data']
[i] Expect to see: training_data.jsonl and fine_tune.py


In [4]:
from pathlib import Path

# Resolve files robustly whether notebook is run from /root/capstone or /root/capstone/scenario_generation
cwd = Path.cwd()
candidates = {
    'training_data.jsonl': [
        cwd / 'training_data.jsonl',
        cwd / 'dataset' / 'training_data.jsonl',
        cwd / 'scenario_generation' / 'dataset' / 'training_data.jsonl',
    ],
    'fine_tune.py': [
        cwd / 'fine_tune.py',
        cwd / 'fine_tuning' / 'fine_tune.py',
        cwd / 'scenario_generation' / 'fine_tuning' / 'fine_tune.py',
    ],
}

resolved = {}
missing = []
for name, paths in candidates.items():
    found = next((p for p in paths if p.exists()), None)
    if found is None:
        missing.append(name)
    else:
        resolved[name] = found.resolve()

if missing:
    raise FileNotFoundError(
        f'Missing required files: {missing}\n'
        f'Current working directory: {cwd}\n'
        'Put files in fine_tuning/ or keep repo structure intact.'
    )

data_path = resolved['training_data.jsonl']
script_path = resolved['fine_tune.py']

print(f'[✓] Found training_data.jsonl at: {data_path}')
print(f'[✓] Found fine_tune.py at: {script_path}')

[✓] Found training_data.jsonl at: /content/training_data.jsonl
[✓] Found fine_tune.py at: /content/fine_tune.py


In [ ]:
# Uses data_path and script_path from the previous cell
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
os.environ["ACCELERATE_DTYPE"] = "float32"

import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# ── Config — based on resolved paths from previous cell ──
DATA_PATH  = str(data_path)
OUTPUT_DIR = "/content/capstone/scenario_generation/fine_tuning/output"
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

# ── Load dataset ──
records = []
with open(DATA_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        if not rec.get("output") or not rec.get("input"):
            continue
        instruction = rec.get("instruction", "")
        text = (
            f"[INST] {instruction}\n\n"
            f"{rec['input']} [/INST]\n"
            f"{rec['output']}"
        )
        records.append({"text": text})

print(f"[+] Loaded {len(records)} records")
dataset = Dataset.from_list(records)
split   = dataset.train_test_split(test_size=0.2, seed=42)
print(f"    Train: {len(split['train'])}  Val: {len(split['test'])}")

# ── Load model in 4-bit ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_quant_storage=torch.float32,
)

print("[+] Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache       = False
model.config.pretraining_tp  = 1

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("[+] Model loaded")

# ── Attach LoRA ──
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"[+] LoRA attached — trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# ── Train ──
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_steps=25,
    save_total_limit=2,
    report_to="none",
    dataset_text_field="text",
    packing=False,
    max_grad_norm=0.3,
)

try:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=split["train"],
        eval_dataset=split["test"],
        args=sft_config,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=split["train"],
        eval_dataset=split["test"],
        args=sft_config,
    )

print("[+] Starting training...")
trainer.train()

print("[+] Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"[✓] Done — saved to {OUTPUT_DIR}")

[+] Loaded 81 records
    Train: 64  Val: 17
[+] Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[+] Model loaded
[+] LoRA attached — trainable: 13,631,488 / 7,261,655,040 (0.19%)


Adding EOS to train dataset:   0%|          | 0/64 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/17 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


[+] Starting training...


Step,Training Loss
10,1.515743
20,0.718061
30,0.547097
40,0.464886


[+] Saving model...
[✓] Done — saved to ./output


In [ ]:
import shutil
from google.colab import files

OUTPUT_DIR = "/content/capstone/scenario_generation/fine_tuning/output"
ARCHIVE_BASENAME = "/content/finetuned_adapters"

shutil.make_archive(ARCHIVE_BASENAME, "zip", OUTPUT_DIR)
files.download(f"{ARCHIVE_BASENAME}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>